# H1 proxy: class-sampling fine-tune from the locked RAMP epoch-80 state
Epochs 81-90 of `ramp_scratch_l5_s{0,1}`, same recipe (λ=5, GP, static lr 0.05). Both arms use the same with-replacement sampler; `cls_ctrl` uses uniform class weights, `cls_fb` draws classes with weights from the training-attack robust accuracy (mean weight 1, clipped to [0.5, 1.5]), so the sample count and compute match. Compare the arms to each other, **not** to the epoch-80 baseline (the control also gets 10 extra epochs).

Set `ARM` below and run the notebook once per arm (two notebooks may run concurrently). Seeds 0 and 1 run in parallel, one per GPU, about 2.4 h training plus about 35 min evaluation per arm. If a session stops at the time budget, run it again; it resumes.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
ARM = 'cls_ctrl'   # run one notebook with 'cls_ctrl' and a second, concurrent one with 'cls_fb'
os.environ['TIME_BUDGET_H'] = '10.5'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b claude/read-hand-off-5m8cj2 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

In [ ]:
stage('install', 'bash scripts/ramp_official.sh install')
stage('data', 'bash scripts/ramp_official.sh data')

In [ ]:
stage(f'{ARM}_train', f'bash scripts/ramp_official.sh train2 {ARM} 0 1')
if all(os.path.exists(f'external/ramp/trained_models/{ARM.replace("_","")}_l5_s{s}/ep_90_0.pth') for s in (0, 1)):
    stage(f'{ARM}_eval', f'bash scripts/ramp_official.sh eval2 {ARM} 0 1')
else:
    print('Training paused at the time budget. Run the notebook again to resume.')